# Text Classification with Naive Bayes

This notebook implements a **Multinomial Naive Bayes** sentiment classifier from scratch for the supplied positive and negative review dataset. It follows the assignment tasks: preprocessing, training, prediction, evaluation, and testing.

In [1]:
from collections import Counter
from pathlib import Path
from zipfile import ZipFile
import csv
import math
import random
import re

ZIP_PATH = Path(r'C:/Users/ACER/Downloads/29_05_2026___b0832e1a603465499638d7510bdc0053.zip')
TEXT_PATH = Path(r'C:/Users/ACER/Downloads/29_05_2026___b0832e1a603465499638d7510bdc0053/text classification with NB.txt')

def load_reviews(zip_path=ZIP_PATH, text_path=TEXT_PATH):
    """Load the CSV-formatted reviews from the extracted TXT file or ZIP archive."""
    if text_path.exists():
        raw_data = text_path.read_text(encoding='utf-8')
    elif zip_path.exists():
        with ZipFile(zip_path) as archive:
            file_name = next(name for name in archive.namelist() if name.endswith('.txt'))
            raw_data = archive.read(file_name).decode('utf-8')
    else:
        raise FileNotFoundError('Update ZIP_PATH or TEXT_PATH to the location of the dataset.')

    return list(csv.DictReader(raw_data.splitlines()))

reviews = load_reviews()
print(f'Total reviews: {len(reviews)}')
print(Counter(row['label'] for row in reviews))
reviews[:3]

Total reviews: 100
Counter({'positive': 50, 'negative': 50})


[{'id': '1',
  'text': 'I love this restaurant because the food is delicious.',
  'label': 'positive'},
 {'id': '2',
  'text': 'The customer service was horrible and slow.',
  'label': 'negative'},
 {'id': '3',
  'text': 'This laptop works perfectly for my daily tasks.',
  'label': 'positive'}]

## Task 1: Data preprocessing

`preprocess_text` converts each review to lowercase and replaces characters that are not letters, digits, or whitespace. The result is tokenized with `split()`. No stop-word removal or stemming is used because the assignment only asks for lowercase conversion and removal of non-alphanumeric characters.

In [2]:
def preprocess_text(text):
    """Lowercase text, remove non-alphanumeric characters, and return tokens."""
    clean_text = re.sub(r'[^a-z0-9\s]', ' ', text.lower())
    return clean_text.split()

example = 'The food was DELICIOUS!!! 10/10.'
print('Original:', example)
print('Tokens:  ', preprocess_text(example))

Original: The food was DELICIOUS!!! 10/10.
Tokens:   ['the', 'food', 'was', 'delicious', '10', '10']


## Train/test split

The split is stratified, so both sentiment classes keep the same proportion in training and testing. The fixed random seed makes the result reproducible.

In [3]:
def stratified_train_test_split(rows, test_size=0.20, seed=42):
    grouped = {'positive': [], 'negative': []}
    for row in rows:
        grouped[row['label']].append(row)

    rng = random.Random(seed)
    train_rows, test_rows = [], []
    for label, group in grouped.items():
        group = group.copy()
        rng.shuffle(group)
        split_index = int(len(group) * (1 - test_size))
        train_rows.extend(group[:split_index])
        test_rows.extend(group[split_index:])

    rng.shuffle(train_rows)
    rng.shuffle(test_rows)
    return train_rows, test_rows

train_rows, test_rows = stratified_train_test_split(reviews)
print(f'Training reviews: {len(train_rows)} | Testing reviews: {len(test_rows)}')
print('Training labels:', Counter(row['label'] for row in train_rows))
print('Testing labels: ', Counter(row['label'] for row in test_rows))

Training reviews: 80 | Testing reviews: 20
Training labels: Counter({'positive': 40, 'negative': 40})
Testing labels:  Counter({'negative': 10, 'positive': 10})


## Task 2: Build a Naive Bayes classifier

The classifier uses a bag-of-words representation. For each class it estimates:

- Prior: `P(class) = documents in class / total documents`
- Likelihood with Laplace smoothing: `P(word | class) = (count(word, class) + 1) / (all class tokens + vocabulary size)`

Log probabilities avoid numerical underflow when a review contains many tokens.

In [4]:
def build_naive_bayes_classifier(positive_data, negative_data):
    """Train a Multinomial Naive Bayes model from positive and negative review lists."""
    positive_tokens = [token for review in positive_data for token in preprocess_text(review)]
    negative_tokens = [token for review in negative_data for token in preprocess_text(review)]

    positive_counts = Counter(positive_tokens)
    negative_counts = Counter(negative_tokens)
    vocabulary = set(positive_counts) | set(negative_counts)
    total_documents = len(positive_data) + len(negative_data)

    return {
        'class_document_counts': {'positive': len(positive_data), 'negative': len(negative_data)},
        'class_token_counts': {'positive': positive_counts, 'negative': negative_counts},
        'total_class_tokens': {'positive': len(positive_tokens), 'negative': len(negative_tokens)},
        'vocabulary': vocabulary,
        'vocabulary_size': len(vocabulary),
        'log_priors': {
            'positive': math.log(len(positive_data) / total_documents),
            'negative': math.log(len(negative_data) / total_documents),
        },
    }

positive_train = [row['text'] for row in train_rows if row['label'] == 'positive']
negative_train = [row['text'] for row in train_rows if row['label'] == 'negative']
classifier = build_naive_bayes_classifier(positive_train, negative_train)

print(f"Vocabulary size: {classifier['vocabulary_size']}")
print('Document counts:', classifier['class_document_counts'])
print('Top positive tokens:', classifier['class_token_counts']['positive'].most_common(10))
print('Top negative tokens:', classifier['class_token_counts']['negative'].most_common(10))

Vocabulary size: 269
Document counts: {'positive': 40, 'negative': 40}
Top positive tokens: [('the', 35), ('and', 12), ('this', 9), ('was', 9), ('i', 8), ('is', 8), ('my', 4), ('very', 4), ('useful', 3), ('product', 3)]
Top negative tokens: [('the', 39), ('and', 8), ('was', 6), ('too', 5), ('is', 4), ('this', 3), ('product', 3), ('stopped', 3), ('frustrating', 3), ('machine', 2)]


## Task 3: Predict sentiments

For a review, the predicted class has the larger log posterior score: `log P(class) + Σ log P(token | class)`.

Laplace smoothing is applied to tokens in the **training vocabulary**. A token never seen during training is ignored because the fitted model has no class-specific evidence for it. This is the standard vocabulary handling used by Multinomial Naive Bayes and avoids inventing evidence for out-of-vocabulary words.


In [5]:
def predict_sentiments(classifier, test_data):
    """Preprocess reviews and predict positive/negative labels with Multinomial NB."""
    predictions = []
    vocabulary = classifier['vocabulary']
    vocabulary_size = classifier['vocabulary_size']

    for review in test_data:
        # Required preprocessing is still applied to every test review.
        tokens = [token for token in preprocess_text(review) if token in vocabulary]
        scores = {}

        for label in ('positive', 'negative'):
            score = classifier['log_priors'][label]
            denominator = classifier['total_class_tokens'][label] + vocabulary_size
            token_counts = classifier['class_token_counts'][label]
            for token in tokens:
                probability = (token_counts[token] + 1) / denominator
                score += math.log(probability)
            scores[label] = score

        predictions.append(max(scores, key=scores.get))

    return predictions

sample_reviews = [
    'The food was excellent and the staff were friendly.',
    'The application crashes and the interface is confusing.',
]
sample_predictions = predict_sentiments(classifier, sample_reviews)
for review, prediction in zip(sample_reviews, sample_predictions):
    print(f'{prediction:>8}: {review}')


positive: The food was excellent and the staff were friendly.
negative: The application crashes and the interface is confusing.


## Task 4: Evaluate the classifier

Accuracy is the number of correct predictions divided by the number of test reviews. The function also returns a small confusion-matrix summary to make errors easy to inspect.

In [6]:
def evaluate_classifier(predictions, true_labels):
    """Calculate accuracy and a binary confusion-matrix summary."""
    if len(predictions) != len(true_labels):
        raise ValueError('predictions and true_labels must have the same length.')

    correct = sum(predicted == actual for predicted, actual in zip(predictions, true_labels))
    confusion = {
        'true_positive': sum(p == 'positive' and y == 'positive' for p, y in zip(predictions, true_labels)),
        'true_negative': sum(p == 'negative' and y == 'negative' for p, y in zip(predictions, true_labels)),
        'false_positive': sum(p == 'positive' and y == 'negative' for p, y in zip(predictions, true_labels)),
        'false_negative': sum(p == 'negative' and y == 'positive' for p, y in zip(predictions, true_labels)),
    }
    precision_positive = confusion['true_positive'] / max(1, confusion['true_positive'] + confusion['false_positive'])
    recall_positive = confusion['true_positive'] / max(1, confusion['true_positive'] + confusion['false_negative'])
    f1_positive = 2 * precision_positive * recall_positive / max(1e-12, precision_positive + recall_positive)
    precision_negative = confusion['true_negative'] / max(1, confusion['true_negative'] + confusion['false_negative'])
    recall_negative = confusion['true_negative'] / max(1, confusion['true_negative'] + confusion['false_positive'])
    f1_negative = 2 * precision_negative * recall_negative / max(1e-12, precision_negative + recall_negative)
    return {
        'accuracy': correct / len(true_labels),
        'correct': correct,
        'total': len(true_labels),
        'precision_positive': precision_positive,
        'recall_positive': recall_positive,
        'f1_positive': f1_positive,
        'f1_negative': f1_negative,
        'macro_f1': (f1_positive + f1_negative) / 2,
        'confusion': confusion,
    }


## Task 5: Test all functions on the dataset

The model is trained with 80 reviews and evaluated on the held-out 20 reviews. Running the notebook again gives the same split and result because the random seed is fixed.

In [7]:
test_texts = [row['text'] for row in test_rows]
true_labels = [row['label'] for row in test_rows]
predictions = predict_sentiments(classifier, test_texts)
results = evaluate_classifier(predictions, true_labels)

print(f"Accuracy: {results['accuracy']:.2%} ({results['correct']}/{results['total']})")
print('Confusion matrix summary:', results['confusion'])
print(f"Positive precision: {results['precision_positive']:.2%}")
print(f"Positive recall:    {results['recall_positive']:.2%}")
print(f"Positive F1-score:  {results['f1_positive']:.2%}")
print(f"Negative F1-score:  {results['f1_negative']:.2%}")
print(f"Macro F1-score:     {results['macro_f1']:.2%}")

print('\nTest-set predictions:')
for row, predicted in zip(test_rows, predictions):
    marker = '[OK]' if predicted == row['label'] else '[WRONG]'
    print(f"{marker} true={row['label']:<8} predicted={predicted:<8} | {row['text']}")

Accuracy: 65.00% (13/20)
Confusion matrix summary: {'true_positive': 9, 'true_negative': 4, 'false_positive': 6, 'false_negative': 1}
Positive precision: 60.00%
Positive recall:    90.00%
Positive F1-score:  72.00%
Negative F1-score:  53.33%
Macro F1-score:     62.67%

Test-set predictions:
[WRONG] true=negative predicted=positive | The microphone quality is very bad.
[OK] true=positive predicted=positive | The vacation was relaxing and enjoyable.
[OK] true=positive predicted=positive | This course is very informative and easy to follow.
[OK] true=positive predicted=positive | The teacher explained the lesson very clearly.
[OK] true=positive predicted=positive | The shoes fit perfectly and look great.
[OK] true=positive predicted=positive | I appreciate the friendly customer support.
[OK] true=negative predicted=negative | The headphones stopped charging yesterday.
[WRONG] true=positive predicted=negative | The weather was perfect during the trip.
[OK] true=negative predicted=negative 

## Improvement: Compare Bernoulli Naive Bayes

The original Multinomial model counts every word occurrence. Bernoulli Naive Bayes uses only whether a word appears in a review. This can help for short reviews, where word presence may matter more than repeated frequency. We compare it on the same holdout set, then use 5-fold cross-validation for a more reliable decision.

In [8]:
def build_bernoulli_naive_bayes_classifier(positive_data, negative_data, alpha=1.0):
    """Train Bernoulli Naive Bayes using binary word-presence features."""
    positive_documents = [set(preprocess_text(review)) for review in positive_data]
    negative_documents = [set(preprocess_text(review)) for review in negative_data]
    positive_counts = Counter(token for document in positive_documents for token in document)
    negative_counts = Counter(token for document in negative_documents for token in document)
    vocabulary = set(positive_counts) | set(negative_counts)

    return {
        'class_document_counts': {'positive': len(positive_documents), 'negative': len(negative_documents)},
        'class_token_counts': {'positive': positive_counts, 'negative': negative_counts},
        'vocabulary': vocabulary,
        'alpha': alpha,
    }

def predict_bernoulli_sentiments(classifier, test_data):
    """Predict labels with Bernoulli Naive Bayes."""
    predictions = []
    vocabulary = classifier['vocabulary']
    alpha = classifier['alpha']
    total_documents = sum(classifier['class_document_counts'].values())

    for review in test_data:
        present_tokens = set(preprocess_text(review))
        scores = {}
        for label in ('positive', 'negative'):
            class_documents = classifier['class_document_counts'][label]
            denominator = class_documents + 2 * alpha
            score = math.log(class_documents / total_documents)
            for token in vocabulary:
                probability = (classifier['class_token_counts'][label][token] + alpha) / denominator
                score += math.log(probability if token in present_tokens else 1 - probability)
            scores[label] = score
        predictions.append(max(scores, key=scores.get))
    return predictions

bernoulli_classifier = build_bernoulli_naive_bayes_classifier(positive_train, negative_train)
bernoulli_predictions = predict_bernoulli_sentiments(bernoulli_classifier, test_texts)
bernoulli_results = evaluate_classifier(bernoulli_predictions, true_labels)
print(f"Multinomial holdout accuracy: {results['accuracy']:.2%}")
print(f"Bernoulli holdout accuracy:   {bernoulli_results['accuracy']:.2%}")

Multinomial holdout accuracy: 65.00%
Bernoulli holdout accuracy:   70.00%


## Improved evaluation: stratified 5-fold cross-validation

A single 20-review test set is small and can give an unstable score. Five-fold cross-validation evaluates every review once while preserving the positive/negative balance in each fold. The final model choice should use the average validation accuracy.

In [9]:
def stratified_folds(rows, n_splits=5, seed=42):
    grouped = {'positive': [], 'negative': []}
    for row in rows:
        grouped[row['label']].append(row)
    rng = random.Random(seed)
    folds = [[] for _ in range(n_splits)]
    for group in grouped.values():
        group = group.copy()
        rng.shuffle(group)
        for index, row in enumerate(group):
            folds[index % n_splits].append(row)
    return folds

def cross_validate(rows, model_type='multinomial', n_splits=5):
    folds = stratified_folds(rows, n_splits=n_splits)
    fold_scores = []
    for fold_index, validation_rows in enumerate(folds):
        training_rows = [row for index, fold in enumerate(folds) if index != fold_index for row in fold]
        positive_data = [row['text'] for row in training_rows if row['label'] == 'positive']
        negative_data = [row['text'] for row in training_rows if row['label'] == 'negative']
        validation_texts = [row['text'] for row in validation_rows]
        validation_labels = [row['label'] for row in validation_rows]

        if model_type == 'multinomial':
            model = build_naive_bayes_classifier(positive_data, negative_data)
            predictions = predict_sentiments(model, validation_texts)
        elif model_type == 'bernoulli':
            model = build_bernoulli_naive_bayes_classifier(positive_data, negative_data)
            predictions = predict_bernoulli_sentiments(model, validation_texts)
        else:
            raise ValueError("model_type must be 'multinomial' or 'bernoulli'.")

        fold_scores.append(evaluate_classifier(predictions, validation_labels)['accuracy'])
    return fold_scores

multinomial_cv = cross_validate(reviews, 'multinomial')
bernoulli_cv = cross_validate(reviews, 'bernoulli')
print('Multinomial fold accuracies:', [f'{score:.0%}' for score in multinomial_cv])
print(f"Multinomial mean accuracy: {sum(multinomial_cv) / len(multinomial_cv):.2%}")
print('Bernoulli fold accuracies:  ', [f'{score:.0%}' for score in bernoulli_cv])
print(f"Bernoulli mean accuracy:   {sum(bernoulli_cv) / len(bernoulli_cv):.2%}")

selected_model = 'multinomial' if sum(multinomial_cv) >= sum(bernoulli_cv) else 'bernoulli'
print(f'Selected model from cross-validation: {selected_model}')

Multinomial fold accuracies: ['70%', '65%', '75%', '85%', '80%']
Multinomial mean accuracy: 75.00%
Bernoulli fold accuracies:   ['75%', '75%', '75%', '70%', '75%']
Bernoulli mean accuracy:   74.00%
Selected model from cross-validation: multinomial


## Comparison and error analysis

The holdout score alone can be misleading because it contains only 20 reviews. The following cell compares the results and measures the out-of-vocabulary (OOV) rate. An OOV token is a word in the test review that did not occur in the training vocabulary.

In [10]:
comparison = [
    ('Multinomial NB', results['accuracy'], results['macro_f1'], sum(multinomial_cv) / len(multinomial_cv)),
    ('Bernoulli NB', bernoulli_results['accuracy'], bernoulli_results['macro_f1'], sum(bernoulli_cv) / len(bernoulli_cv)),
]
print(f"{'Model':<18} {'Holdout accuracy':>18} {'Holdout macro-F1':>19} {'5-fold mean accuracy':>22}")
for model_name, holdout_accuracy, holdout_f1, cv_accuracy in comparison:
    print(f"{model_name:<18} {holdout_accuracy:>17.2%} {holdout_f1:>18.2%} {cv_accuracy:>21.2%}")

def oov_rate(review, vocabulary):
    tokens = preprocess_text(review)
    return sum(token not in vocabulary for token in tokens) / len(tokens) if tokens else 0

correct_oov = []
incorrect_oov = []
for row, predicted in zip(test_rows, predictions):
    rate = oov_rate(row['text'], classifier['vocabulary'])
    (correct_oov if predicted == row['label'] else incorrect_oov).append(rate)

print(f"\nAverage OOV rate for correct predictions:   {sum(correct_oov) / len(correct_oov):.2%}")
print(f"Average OOV rate for incorrect predictions: {sum(incorrect_oov) / len(incorrect_oov):.2%}")

print('\nErrors with the highest OOV rate:')
errors = [(oov_rate(row['text'], classifier['vocabulary']), row['label'], predicted, row['text'])
          for row, predicted in zip(test_rows, predictions) if predicted != row['label']]
for rate, actual, predicted, review in sorted(errors, reverse=True)[:5]:
    print(f"OOV={rate:.0%} | true={actual:<8} predicted={predicted:<8} | {review}")

Model                Holdout accuracy    Holdout macro-F1   5-fold mean accuracy
Multinomial NB                65.00%             62.67%                75.00%
Bernoulli NB                  70.00%             69.70%                74.00%

Average OOV rate for correct predictions:   39.57%
Average OOV rate for incorrect predictions: 38.62%

Errors with the highest OOV rate:
OOV=50% | true=negative predicted=positive | The instructions are unclear and incomplete.
OOV=43% | true=positive predicted=negative | The weather was perfect during the trip.
OOV=40% | true=negative predicted=positive | The audio quality is distorted.
OOV=38% | true=negative predicted=positive | The interface is confusing and difficult to use.
OOV=33% | true=negative predicted=positive | The software update deleted my files.


## Confidence-aware prediction (optional extension)

Tasks 1–5 remain unchanged: `predict_sentiments` always returns `positive` or `negative`. This optional deployment layer prevents the system from forcing a label when the two class scores are too close.

The absolute difference between the two log scores is used as a confidence margin. It is useful for ranking confidence, but it is **not a calibrated probability**. Reporting it as a true probability would overstate the model's certainty.


In [11]:
def predict_with_margin(classifier, review):
    vocabulary = classifier['vocabulary']
    tokens = [token for token in preprocess_text(review) if token in vocabulary]
    vocabulary_size = classifier['vocabulary_size']
    scores = {}
    for label in ('positive', 'negative'):
        score = classifier['log_priors'][label]
        denominator = classifier['total_class_tokens'][label] + vocabulary_size
        token_counts = classifier['class_token_counts'][label]
        for token in tokens:
            score += math.log((token_counts[token] + 1) / denominator)
        scores[label] = score
    prediction = max(scores, key=scores.get)
    margin = abs(scores['positive'] - scores['negative'])
    return prediction, margin, scores

def safe_predict_sentiments(classifier, test_data, threshold=1.00):
    """Optional extension: abstain when the log-score margin is too small."""
    results = []
    for review in test_data:
        prediction, margin, scores = predict_with_margin(classifier, review)
        results.append({
            'sentiment': prediction if margin >= threshold else 'uncertain',
            'raw_prediction': prediction,
            'margin': margin,
            'scores': scores,
        })
    return results

safe_examples = safe_predict_sentiments(classifier, [
    'The product is excellent and works perfectly.',
    'The interface is okay.',
    'The service was horrible and extremely slow.',
])
for result in safe_examples:
    print(result)


{'sentiment': 'positive', 'raw_prediction': 'positive', 'margin': 1.8420666378627395, 'scores': {'positive': -21.141297866284916, 'negative': -22.983364504147655}}
{'sentiment': 'positive', 'raw_prediction': 'positive', 'margin': 1.111555381699251, 'scores': {'positive': -12.98650660946777, 'negative': -14.098061991167022}}
{'sentiment': 'negative', 'raw_prediction': 'negative', 'margin': 1.6097775455185754, 'scores': {'positive': -35.339196903361874, 'negative': -33.7294193578433}}


## Reliability evaluation: repeated cross-validation

Selective accuracy measures correctness only on reviews for which the model produces a positive or negative label. Coverage is the proportion of all reviews that receive a label; the rest are marked `uncertain`. Both values must always be reported together.

Because this dataset has only 100 reviews, one five-fold split can be unusually easy or difficult. The evaluation below therefore repeats stratified five-fold cross-validation with 20 seeds (2,000 out-of-fold predictions). This gives a more stable estimate and prevents choosing a threshold from one lucky split.


In [12]:
def cross_validated_margin_records(rows, n_splits=5, seed=42):
    folds = stratified_folds(rows, n_splits=n_splits, seed=seed)
    records = []
    for fold_index, validation_rows in enumerate(folds):
        training_rows = [row for index, fold in enumerate(folds) if index != fold_index for row in fold]
        positive_data = [row['text'] for row in training_rows if row['label'] == 'positive']
        negative_data = [row['text'] for row in training_rows if row['label'] == 'negative']
        model = build_naive_bayes_classifier(positive_data, negative_data)
        for row in validation_rows:
            prediction, margin, _ = predict_with_margin(model, row['text'])
            records.append({
                'prediction': prediction,
                'true_label': row['label'],
                'margin': margin,
                'text': row['text'],
                'seed': seed,
            })
    return records

def evaluate_selective_predictions(records, threshold):
    accepted = [record for record in records if record['margin'] >= threshold]
    correct = sum(record['prediction'] == record['true_label'] for record in accepted)
    return {
        'threshold': threshold,
        'selective_accuracy': correct / len(accepted) if accepted else 0.0,
        'coverage': len(accepted) / len(records),
        'accepted': len(accepted),
        'abstained': len(records) - len(accepted),
    }

# Repeat five-fold CV over 20 deterministic shuffles.
REPEATED_SEEDS = range(20)
repeated_margin_records = [
    record
    for seed in REPEATED_SEEDS
    for record in cross_validated_margin_records(reviews, n_splits=5, seed=seed)
]

forced_accuracy = sum(
    r['prediction'] == r['true_label'] for r in repeated_margin_records
) / len(repeated_margin_records)

seed_accuracies = []
for seed in REPEATED_SEEDS:
    seed_records = [r for r in repeated_margin_records if r['seed'] == seed]
    seed_accuracies.append(
        sum(r['prediction'] == r['true_label'] for r in seed_records) / len(seed_records)
    )
mean_seed_accuracy = sum(seed_accuracies) / len(seed_accuracies)
seed_variance = sum(
    (score - mean_seed_accuracy) ** 2 for score in seed_accuracies
) / (len(seed_accuracies) - 1)
confidence_half_width = 1.96 * math.sqrt(seed_variance / len(seed_accuracies))

print(f'Repeated forced-prediction accuracy: {forced_accuracy:.2%}')
print(
    'Approximate 95% CI across split seeds: '
    f'{mean_seed_accuracy - confidence_half_width:.2%} to '
    f'{mean_seed_accuracy + confidence_half_width:.2%}'
)
print(f"{'Threshold':>10} {'Accuracy':>12} {'Coverage':>12} {'Accepted':>10} {'Uncertain':>11}")
for threshold in (0.0, 0.5, 0.6, 0.75, 1.0, 1.25, 1.5, 2.0):
    report = evaluate_selective_predictions(repeated_margin_records, threshold)
    print(f"{threshold:>10.2f} {report['selective_accuracy']:>11.2%} {report['coverage']:>11.2%} {report['accepted']:>10} {report['abstained']:>11}")

# Chosen before deployment as a practical reliability/coverage trade-off.
SAFE_THRESHOLD = 1.00
safe_report = evaluate_selective_predictions(repeated_margin_records, SAFE_THRESHOLD)
print(f'\nSelected threshold: {SAFE_THRESHOLD:.2f}')
print(f"Repeated-CV selective accuracy: {safe_report['selective_accuracy']:.2%}")
print(f"Coverage: {safe_report['coverage']:.2%}")


Repeated forced-prediction accuracy: 71.55%
Approximate 95% CI across split seeds: 70.67% to 72.43%
 Threshold     Accuracy     Coverage   Accepted   Uncertain
      0.00      71.55%     100.00%       2000           0
      0.50      78.92%      64.75%       1295         705
      0.60      80.79%      59.60%       1192         808
      0.75      82.52%      53.50%       1070         930
      1.00      86.62%      41.85%        837        1163
      1.25      87.37%      34.45%        689        1311
      1.50      88.24%      28.05%        561        1439
      2.00      88.86%      17.05%        341        1659

Selected threshold: 1.00
Repeated-CV selective accuracy: 86.62%
Coverage: 41.85%


## Final deployment model

After model selection and evaluation, the final classifier is trained on all 100 labeled reviews. Production predictions use the selected threshold and return `uncertain` when evidence is weak. This reduces forced errors, but it does not guarantee that every accepted prediction is correct.

In [13]:
all_positive_data = [row['text'] for row in reviews if row['label'] == 'positive']
all_negative_data = [row['text'] for row in reviews if row['label'] == 'negative']
final_classifier = build_naive_bayes_classifier(all_positive_data, all_negative_data)

def classify_review_safely(review):
    return safe_predict_sentiments(final_classifier, [review], threshold=SAFE_THRESHOLD)[0]

print(classify_review_safely('The camera is fantastic and easy to use.'))
print(classify_review_safely('The device is acceptable.'))

{'sentiment': 'positive', 'raw_prediction': 'positive', 'margin': 3.7389495866920868, 'scores': {'positive': -38.39859118167237, 'negative': -42.13754076836446}}
{'sentiment': 'uncertain', 'raw_prediction': 'positive', 'margin': 0.13445209504058298, 'scores': {'positive': -7.30565804812419, 'negative': -7.440110143164773}}


## Final model choice and compliance with the assignment

The notebook still satisfies every original task:

1. `preprocess_text(text)` lowercases text and removes non-alphanumeric characters.
2. `build_naive_bayes_classifier(positive_data, negative_data)` preprocesses the reviews and trains Multinomial Naive Bayes.
3. `predict_sentiments(classifier, test_data)` preprocesses test reviews and always predicts `positive` or `negative`.
4. `evaluate_classifier(predictions, true_labels)` calculates accuracy and also reports diagnostic metrics.
5. All functions are applied to the supplied reviews and evaluated on held-out and cross-validated data.

The only correction to the required classifier is standard out-of-vocabulary handling: unseen test tokens are ignored because they provide no learned class evidence. Bigram, TF-IDF, character n-gram, Complement NB, and Bernoulli NB variants were evaluated with repeated stratified cross-validation but did not outperform unigram Multinomial NB consistently.

Under repeated stratified five-fold evaluation across 20 seeds, the corrected required classifier achieves **71.55% forced-prediction accuracy** with an approximate split-seed 95% interval of **70.67%–72.43%**. The optional threshold of **1.00** reaches **86.62% selective accuracy** at **41.85% coverage**. Compared with the earlier non-standard OOV treatment, these figures improve from 71.25% and 86.22%, respectively.

The `uncertain` result is an optional reliability layer and does not replace the required Task 3 predictions. Accuracy and coverage must be reported together. With only 100 reviews, results remain preliminary; more labeled data and an independent final test set are needed before making production claims.
